### Velocity Computation and Coarse Graining

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from multiprocessing import Pool, cpu_count
import gzip
import re
from scipy.spatial import cKDTree
import torch

class VelocityCoarseGrainer:
    """
    Goldhirsch-Weinhart coarse-graining implementation for DEM Simulation data.
    
    Velocities are computed via finite differences of particle positions between
    consecutive dump frames:
        v_i = (x_{i+1} - x_i) / delta_t
    
    The first frame (index 0) is assigned zero velocity (initially at rest).
    If 1001 frames are read, 1000 FD velocities are computed; frame 0 gets v=0,
    frames 1..1000 get the FD velocity computed from the preceding pair.
    """
    BULK_TYPE   = 3      # LAMMPS atom type for granular (fluid) particles

    def __init__(self, folder_path, pattern='dump.stress.*',
                 max_frames=np.inf, n_cores=None,
                 dt=5.18e-6):
        """
        Parameters
        ----------
        folder_path : str | Path
        pattern : str
            Glob pattern for dump files.
        max_frames : int | float
            Maximum number of frames to load.
        n_cores : int | None
            Number of CPU cores for parallel work.
        dt : float
            Physical time represented by one LAMMPS timestep (seconds).
            delta_t between two frames = (timestep_j - timestep_i) * dt.
        timestep_to_seconds : float
            Alias for dt; if both are given dt takes precedence.
            Kept for backwards compatibility.
        """
        self.folder_path = Path(folder_path)
        self.pattern = pattern
        self.max_frames = max_frames
        self.n_cores = n_cores or cpu_count()
        self.dt = dt  # physical seconds per LAMMPS step

        # Particle type densities (kg/m³)
        self.type_density = {1: 2600.0, 2: 2600.0, 3: 2600.0}

        # Coarse-graining parameters
        self.w = None
        self.support_fac = 3.0
        self.support = None
        self.dx = None

        print(f"Coarse-grainer initialized with folder: {self.folder_path}")
        print(f"Physical dt per LAMMPS step: {self.dt}")

    # ------------------------------------------------------------------
    # Kernel
    # ------------------------------------------------------------------

    def gaussian_kernel(self, r, w):
        """3-D Gaussian kernel φ(r) = (1/√(2πw²))³ exp(-r²/(2w²))"""
        norm = 1.0 / (np.sqrt(2 * np.pi) * w) ** 3
        return norm * np.exp(-r ** 2 / (2 * w ** 2))

    # ------------------------------------------------------------------
    # File discovery & reading
    # ------------------------------------------------------------------

    def find_and_sort_files(self):
        """Find and sort LAMMPS dump files by timestep."""
        print(f"Looking for files in: {self.folder_path}")

        if not self.folder_path.exists():
            raise FileNotFoundError(f"Folder does not exist: {self.folder_path}")

        files = list(self.folder_path.glob(self.pattern))
        if not files:
            raise FileNotFoundError(f"No files found matching pattern: {self.pattern}")

        print(f"Found {len(files)} files matching pattern")

        patterns = [
            r'\.(\d+)$', r'\.(\d+)\.gz$', r'\.(\d+)\..*$',
            r'wall\.(\d+)', r'_(\d+)$', r'(\d+)$'
        ]

        timesteps, valid_files = [], []
        for file in files:
            for pat in patterns:
                match = re.search(pat, file.name)
                if match:
                    timesteps.append(int(match.group(1)))
                    valid_files.append(file)
                    break

        if not valid_files:
            raise ValueError("No files with extractable timesteps found")

        sorted_indices = np.argsort(timesteps)
        self.files = [valid_files[i] for i in sorted_indices]
        self.timesteps = [timesteps[i] for i in sorted_indices]

        print(f"Processing {len(self.files)} files with valid timesteps")
        return self.files[:int(self.max_frames)]

    def read_lammps_dump(self, filename):
        """Read a single LAMMPS dump file."""
        filepath = Path(filename)
        opener = gzip.open if filepath.suffix == '.gz' else open

        with opener(filepath, 'rt') as f:
            line = f.readline()
            if not line.startswith('ITEM: TIMESTEP'):
                raise ValueError("Expected TIMESTEP header")
            timestep = int(f.readline().strip())

            line = f.readline()
            if not line.startswith('ITEM: NUMBER OF ATOMS'):
                raise ValueError("Expected NUMBER OF ATOMS header")
            N = int(f.readline().strip())

            line = f.readline()
            if not line.startswith('ITEM: BOX BOUNDS'):
                raise ValueError("Expected BOX BOUNDS header")
            boundary_types = line.split()[3:6] if len(line.split()) > 3 else ['pp', 'pp', 'pp']

            bounds = []
            for _ in range(3):
                line = f.readline().strip()
                if line.startswith('ITEM:'):
                    break
                vals = list(map(float, line.split()))
                if len(vals) >= 2:
                    bounds.append(vals[:2])

            bounds = np.array(bounds) 
            dim = len(bounds)

            if not line.startswith('ITEM: ATOMS'):
                line = f.readline()
            cols = line.split()[2:]

            data = []
            for _ in range(N):
                line = f.readline().strip()
                if line:
                    data.append(list(map(float, line.split())))
            data = np.array(data)

        frame = {
            'timestep': timestep,
            'N': N,
            'box_bounds': bounds,
            'boundary_types': boundary_types,
            'dim': dim,
            'x': np.zeros((N, max(2, dim))),
            # 'v' here stores the dump-file velocities (kept for reference).
            # Finite-difference velocities are stored in 'v_fd' after calling
            # compute_finite_difference_velocities().
            'v': np.zeros((N, max(2, dim))),
            'type': np.ones(N, dtype=int),
            'radius': np.zeros(N) * 0.005,
            'mass': None,
            'id': None,
        }
        frame['radius'] = self.read_radius_from_config(
            r"F:\DEM_DATA\const_V_3D\vf601vt03\config_601.txt"
        )

        for i, col in enumerate(cols):
            col = col.lower()
            if col == 'id':
                frame['id'] = data[:, i].astype(int)
            elif col == 'type':
                frame['type'] = data[:, i].astype(int)
            elif col in ['x', 'xu']:
                frame['x'][:, 0] = data[:, i]
            elif col in ['y', 'yu']:
                frame['x'][:, 1] = data[:, i]
            elif col in ['z', 'zu'] and dim >= 3:
                frame['x'][:, 2] = data[:, i]
            elif col == 'vx':
                frame['v'][:, 0] = data[:, i]
            elif col == 'vy':
                frame['v'][:, 1] = data[:, i]
            elif col == 'vz' and dim >= 3:
                frame['v'][:, 2] = data[:, i]
            elif col in ['mass', 'm']:
                frame['mass'] = data[:, i]

        return frame

    def read_all_frames(self):
        """
        Read all LAMMPS dump files and compute finite-difference velocities.

        After this call every frame has a 'v_fd' key (shape N x dim) containing
        the position-derived velocity:
            frame[0]['v_fd'] = 0   (at rest initially)
            frame[i]['v_fd'] = (x_{i} - x_{i-1}) / delta_t   for i >= 1
        """
        files = self.find_and_sort_files()
        frames = []

        for i, file in enumerate(files):
            try:
                frame = self.read_lammps_dump(file)
                frames.append(frame)
            except Exception as e:
                print(f"  ✗ Error reading {file.name}: {e}")
                import traceback
                traceback.print_exc()

        if not frames:
            raise RuntimeError("No frames successfully read")

        print(f"\n✓ Successfully read {len(frames)} frames")
        self.frames = frames

        # Compute finite-difference velocities across all frames
        self.compute_finite_difference_velocities(frames)

        return frames

    # ------------------------------------------------------------------
    # Finite-difference velocity computation  ← NEW
    # ------------------------------------------------------------------

    def compute_finite_difference_velocities(self, frames):
        """
        Compute per-particle velocities from finite differences of positions.

        Stores results in frame['v_fd'] for every frame.

        Convention
        ----------
        frame[0]['v_fd']  = zeros  (system at rest before first step)
        frame[i]['v_fd']  = (x_i - x_{i-1}) / delta_t   for i = 1 … N_frames-1

        where delta_t = (timestep_i - timestep_{i-1}) * self.dt  (physical time).

        Notes
        -----
        - Particle ordering must be consistent across frames.  If dump files
          contain particle IDs ('id' column) the positions are re-ordered by ID
          before differencing so that x[j] always refers to the same particle.
        - Periodic-boundary unwrapping is attempted automatically when the
          displacement magnitude exceeds half the box length in any direction.
        """
        print("\nComputing finite-difference velocities …")

        n_frames = len(frames)
        dim = frames[0]['dim']

        # --- sort particles by ID inside each frame (if IDs are present) ---
        for frame in frames:
            if frame['id'] is not None:
                order = np.argsort(frame['id'])
                frame['x'] = frame['x'][order]
                frame['v'] = frame['v'][order]
                frame['type'] = frame['type'][order]
                frame['radius'] = frame['radius'][order]
                if frame['mass'] is not None:
                    frame['mass'] = frame['mass'][order]
                frame['id'] = frame['id'][order]

        # --- frame 0: zero velocity ---
        N = frames[0]['N']
        frames[0]['v_fd'] = np.zeros((N, dim))
        dt = 0.0000051868 #time step in lammps simulation in seconds
        # --- frames 1 … n_frames-1 ---
        for i in range(1, n_frames):
            prev = frames[i - 1]
            curr = frames[i]

            # Physical time elapsed between the two frames
            delta_lammps_steps = curr['timestep'] - prev['timestep']
            delta_t = delta_lammps_steps * dt

            if delta_t == 0:
                print(f"  ⚠ Frame {i}: delta_t = 0, setting v_fd = 0")
                curr['v_fd'] = np.zeros((curr['N'], dim))
                continue

            # Raw displacement
            dx = curr['x'][:, :dim] - prev['x'][:, :dim]

            # Periodic boundary unwrapping
            box_lengths = curr['box_bounds'][:dim, 1] - curr['box_bounds'][:dim, 0]
            for d in range(dim):
                btype = curr['boundary_types'][d] if d < len(curr['boundary_types']) else 'pp'
                if 'p' in btype:           # periodic in this direction
                    L = box_lengths[d]
                    dx[:, d] -= L * np.round(dx[:, d] / L)

            curr['v_fd'] = dx / delta_t

            if i % 100 == 0 or i == n_frames - 1:
                print(f"  Frame {i}/{n_frames - 1}" f"|v_fd| max = {np.max(np.linalg.norm(curr['v_fd'], axis=1)):.4g}")

        print(f"✓ Finite-difference velocities computed for {n_frames} frames "
              f"({n_frames - 1} non-zero + 1 zero-initialised)")

    # ------------------------------------------------------------------
    # Coarse-graining (uses v_fd by default)
    # ------------------------------------------------------------------

    def compute_particle_mass(self, frame):
        """Compute particle masses if not provided."""
        if frame['mass'] is not None:
            return frame['mass']
        volume = (4.0 / 3.0) * np.pi * frame['radius'] ** 3
        return np.array([volume[i] * self.type_density.get(frame['type'][i], 2600.0)
                         for i in range(frame['N'])])

    def read_radius_from_config(self, config_file):
        """Read radii from a LAMMPS config file (diameter column)."""
        with open(config_file, 'r') as f:
            lines = f.readlines()

        particles_start = None
        for i, line in enumerate(lines):
            if 'Atoms' in line:
                particles_start = i + 2
                break

        if particles_start is None:
            raise ValueError("Atoms section not found in config file")

        particle_data = []
        for line in lines[particles_start:]:
            if not line.strip():
                break
            parts = line.split()
            if len(parts) >= 3:
                particle_data.append((int(parts[0]), float(parts[2])))

        particle_data.sort(key=lambda x: x[0])
        diameters = np.array([d for _, d in particle_data])
        return diameters / 2.0

    def estimate_coarse_graining_width(self, frame, xi_mode='particle'):
        """
        Compute CG kernel width ξ.

        Parameters
        ----------
        xi_mode : 'particle' (default) or 'smooth'
            'particle'  ξ = 2·dp = 2R̄
                IKH near-wall profile mode.  Resolves layering at 1dp.
                erf Z(y) correction essential for first ~2 wall nodes.

            'smooth'    ξ = 3·dp = 6R̄
                Bulk diagnostic mode.  Suppresses force-chain noise so
                that ‖∇·σ‖ diagnostics are reliable.  Correction needed
                for first ~9 nodes near wall.
        """
        mean_r = np.mean(frame['radius'])
        dp     = 2.0 * mean_r

        if xi_mode == 'particle':
            w     = 2.0 * dp
            label = "2·dp  (particle-scale IKH)"
        elif xi_mode == 'smooth':
            w     = 3.0 * dp
            label = "3·dp  (smooth, momentum-balance)"
        else:
            raise ValueError(f"xi_mode must be 'particle' or 'smooth', got '{xi_mode}'")

        print(f"Coarse-graining width:")
        print(f"  Mean radius R̄   : {mean_r:.6f} m")
        print(f"  Particle diam dp : {dp:.6f} m")
        print(f"  ξ = {label} : {w:.6f} m")
        print(f"  Support {self.support_fac}·ξ     : {self.support_fac * w:.6f} m")
        print(f"  erf correction needed for first ~{self.support_fac * w / dp:.1f} nodes from wall")
        return w
    

    def coarse_grain_velocity(self, frame, grid_points, w=None, use_fd_velocity=True):
        """
        Coarse-grain velocity field using Goldhirsch-Weinhart method.

        Parameters
        ----------
        frame : dict
            Frame data (must contain 'v_fd' if use_fd_velocity=True).
        grid_points : ndarray  (N_grid × dim)
        w : float | None
            Coarse-graining width; auto-estimated if None.
        use_fd_velocity : bool
            If True (default) use finite-difference velocities ('v_fd').
            If False use the velocities stored in the dump file ('v').

        Returns
        -------
        velocity_field : ndarray  (N_grid × dim)
        density_field  : ndarray  (N_grid,)
        """
        if w is None:
            w = self.estimate_coarse_graining_width(frame)
        else:
            print(f"Using provided coarse-graining width: {w}")

        support = self.support_fac * w
        dim = frame['dim']
        positions = frame['x'][:, :dim]

        # Choose velocity source
        if use_fd_velocity:
            if 'v_fd' not in frame:
                raise KeyError("'v_fd' not found in frame. "
                               "Call compute_finite_difference_velocities() first "
                               "or set use_fd_velocity=False.")
            velocities = frame['v_fd'][:, :dim]
            vel_label = "finite-difference"
        else:
            velocities = frame['v'][:, :dim]
            vel_label = "dump-file"

        print(f"Velocity source: {vel_label}")

        mass = self.compute_particle_mass(frame)
        tree = cKDTree(positions)

        n_grid = len(grid_points)
        velocity_field = np.zeros((n_grid, dim))
        density_field = np.zeros(n_grid)

        print(f"Coarse-graining {n_grid} grid points …")
        for i, grid_pt in enumerate(grid_points):
            if i % 1000 == 0:
                print(f"  Progress: {i}/{n_grid} points")

            indices = tree.query_ball_point(grid_pt[:dim], support)
            if not indices:
                continue

            r_vec = positions[indices] - grid_pt[:dim]
            r = np.linalg.norm(r_vec, axis=1)
            weights = self.gaussian_kernel(r, w) * mass[indices]

            total_weight = np.sum(weights)
            if total_weight > 1e-12:
                velocity_field[i] = (np.sum(weights[:, None] * velocities[indices], axis=0)
                                     / total_weight)
                density_field[i] = total_weight

        print("  ✓ Coarse-graining complete")
        return velocity_field, density_field

    def compute_velocity_profile(self, frame, direction='y', velocity_component='x',
                                 n_bins=50, w=None, use_fd_velocity=True):
        """
        Compute 1D velocity profile along a direction (useful for shear flows).

        Parameters
        ----------
        frame : dict
        direction : str | int
            Direction to bin along ('x', 'y', 'z' or 0, 1, 2).
        velocity_component : str | int
            Velocity component to average.
        n_bins : int
        w : float | None
            Coarse-graining width.
        use_fd_velocity : bool
            If True (default) use 'v_fd'; otherwise use dump-file 'v'.

        Returns
        -------
        bin_centers     : ndarray
        velocity_profile: ndarray
        velocity_std    : ndarray
        particle_count  : ndarray
        """
        dir_map = {'x': 0, 'y': 1, 'z': 2}
        dir_idx = dir_map[direction.lower()] if isinstance(direction, str) else direction
        vel_idx = (dir_map[velocity_component.lower()]
                   if isinstance(velocity_component, str) else velocity_component)

        positions = frame['x'][:, dir_idx]

        if use_fd_velocity:
            if 'v_fd' not in frame:
                raise KeyError("'v_fd' not found. Run compute_finite_difference_velocities first.")
            velocities = frame['v_fd'][:, vel_idx]
            vel_label = "finite-difference"
        else:
            velocities = frame['v'][:, vel_idx]
            vel_label = "dump-file"

        if w is None:
            w = self.estimate_coarse_graining_width(frame)

        bounds = frame['box_bounds'][dir_idx]
        bin_edges = np.linspace(bounds[0]+w, bounds[1]-w, n_bins + 1)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        velocity_profile = np.zeros(n_bins)
        velocity_std = np.zeros(n_bins)
        particle_count = np.zeros(n_bins)

        for i, center in enumerate(bin_centers):
            distances = np.abs(positions - center)
            mask = distances < (self.support_fac * w)

            if np.sum(mask) > 0:
                weights = self.gaussian_kernel(distances[mask], w)
                velocity_profile[i] = np.average(velocities[mask], weights=weights)
                velocity_std[i] = np.std(velocities[mask])
                particle_count[i] = np.sum(mask)

        print(f"Velocity profile computed ({vel_label}):")
        print(f"  Direction: {direction} (index {dir_idx})")
        print(f"  Velocity component: {velocity_component} (index {vel_idx})")
        print(f"  Bins with particles: {np.sum(particle_count > 0)}/{n_bins}")

        return bin_centers, velocity_profile, velocity_std, particle_count

    # ------------------------------------------------------------------
    # Grid creation
    # ------------------------------------------------------------------


    def create_grid(self, frame, grid_type='uniform', n_points=None, dx=None):
        bounds = frame['box_bounds']
        dim = frame['dim']

        # 1. Ensure 'w' is always defined for the boundary logic
        w = self.estimate_coarse_graining_width(frame)

        # 2. Handle default dx if nothing is provided
        if n_points is None and dx is None:
            dx = w / 2.0
            print(f"Using grid spacing: {dx:.6f}")

        # 3. Calculate n_points based on dx and bounds
        if n_points is None:
            if np.isscalar(dx):
                dx = np.array([dx] * dim)

            n_pts_list = []
            for i in range(dim):
                # Calculate the effective length for this dimension
                if i == 1: # Apply buffer 'w' to the y-dimension
                    length = (bounds[i, 1] - bounds[i, 0]) - 2 * w
                else:
                    length = (bounds[i, 1] - bounds[i, 0])

                # Ensure we have at least 1 point and convert to int
                n_pts_list.append(max(int(length / dx[i]), 1))

            n_points = tuple(n_pts_list)

        # 4. Handle scalar n_points input
        if isinstance(n_points, int):
            n_points = tuple([n_points] * dim)

        # 5. Generate the grid
        if dim == 2:
            x = np.linspace(bounds[0, 0], bounds[0, 1], n_points[0])
            y = np.linspace(bounds[1, 0] + w, bounds[1, 1] - w, n_points[1])
            xx, yy = np.meshgrid(x, y)
            grid_points = np.column_stack([xx.ravel(), yy.ravel()])
            grid_shape = (n_points[1], n_points[0])
        else:
            x = np.linspace(bounds[0, 0], bounds[0, 1], n_points[0])
            y = np.linspace(bounds[1, 0] + w, bounds[1, 1] - w, n_points[1])
            z = np.linspace(bounds[2, 0], bounds[2, 1], n_points[2])
            # Use indexing='ij' to keep (x, y, z) order consistent
            xx, yy, zz = np.meshgrid(x, y, z, indexing='ij')
            grid_points = np.column_stack([xx.ravel(), yy.ravel(), zz.ravel()])
            grid_shape = n_points

        print(f"Created {grid_type} grid: {n_points} points")
        return grid_points, grid_shape

    # ------------------------------------------------------------------
    # Plotting helpers
    # ------------------------------------------------------------------

    def plot_velocity_profile(self, frame, velocity_field, grid_points, grid_shape,
                              slice_dim=None, slice_val=None, component=0, support=None):
        """Create scatter/image plot of velocity profiles."""
        dim = frame['dim']
        if support is None:
            support = self.estimate_coarse_graining_width(frame)

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        # Left: raw particle velocities (from v_fd if available, else dump v)
        ax1 = axes[0]
        if 'v_fd' in frame:
            particle_vel = frame['v_fd'][:, component]
            ax1.set_title(f'FD Particle Velocities (Component {component})')
        else:
            particle_vel = frame['v'][:, component]
            ax1.set_title(f'Dump Particle Velocities (Component {component})')

        vmin_p = np.nanpercentile(particle_vel, 1)
        vmax_p = np.nanpercentile(particle_vel, 99)

        if dim == 2:
            sc = ax1.scatter(frame['x'][:, 0], frame['x'][:, 1],
                             c=particle_vel, s=1, cmap='viridis',
                             alpha=0.7, vmin=vmin_p, vmax=vmax_p)
        else:
            pts_x, pts_y, vel_plot = frame['x'][:, 0], frame['x'][:, 1], particle_vel
            if slice_dim is not None and slice_val is not None:
                mask = np.abs(frame['x'][:, slice_dim] - slice_val) < support
                pts_x, pts_y, vel_plot = pts_x[mask], pts_y[mask], vel_plot[mask]
            sc = ax1.scatter(pts_x, pts_y, c=vel_plot, s=2, cmap='viridis',
                             alpha=0.7, vmin=vmin_p, vmax=vmax_p)

        ax1.set_xlabel('x')
        ax1.set_ylabel('y')
        plt.colorbar(sc, ax=ax1, label=f'v_{component}')

        # Right: coarse-grained field
        ax2 = axes[1]
        vel_component = velocity_field[:, component]
        valid_mask = ~np.isnan(vel_component) & (vel_component != 0)
        if np.sum(valid_mask) > 0:
            vmin_c = np.percentile(vel_component[valid_mask], 1)
            vmax_c = np.percentile(vel_component[valid_mask], 99)
        else:
            vmin_c, vmax_c = vel_component.min(), vel_component.max()

        if vmin_c == vmax_c:
            vmin_c -= 0.1 * abs(vmin_c) if vmin_c != 0 else 0.1
            vmax_c += 0.1 * abs(vmax_c) if vmax_c != 0 else 0.1

        if dim == 2:
            vel_grid = vel_component.reshape(grid_shape)
            im = ax2.imshow(vel_grid,
                            extent=[frame['box_bounds'][0, 0], frame['box_bounds'][0, 1],
                                    frame['box_bounds'][1, 0], frame['box_bounds'][1, 1]],
                            origin='lower', cmap='viridis', aspect='auto',
                            vmin=vmin_c, vmax=vmax_c, interpolation='bilinear')
            plt.colorbar(im, ax=ax2, label=f'v_{component}')
        else:
            gx, gy = grid_points[:, 0], grid_points[:, 1]
            vc = vel_component
            if slice_dim is not None and slice_val is not None:
                mask = np.abs(grid_points[:, slice_dim] - slice_val) < support
                gx, gy, vc = gx[mask], gy[mask], vc[mask]
            sc2 = ax2.scatter(gx, gy, c=vc, s=50, cmap='viridis',
                              vmin=vmin_c, vmax=vmax_c, edgecolors='none')
            plt.colorbar(sc2, ax=ax2, label=f'v_{component}')

        ax2.set_xlabel('x')
        ax2.set_ylabel('y')
        ax2.set_title(f'Coarse-Grained Velocity Field (Component {component})')

        plt.tight_layout()
        plt.show()
        return fig

    def plot_1d_velocity_profile(self, bin_centers, velocity_profile, velocity_std=None,
                                 particle_count=None, direction='y', velocity_component='x'):
        """Plot 1D velocity profile."""
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        ax1 = axes[0]
        ax1.plot(bin_centers, velocity_profile, 'b-', linewidth=2, label='Coarse-grained')
        if velocity_std is not None:
            ax1.fill_between(bin_centers,
                             velocity_profile - velocity_std,
                             velocity_profile + velocity_std,
                             alpha=0.3, label='±1 std dev')
        ax1.set_xlabel(f'{direction} position')
        ax1.set_ylabel(f'v_{velocity_component}')
        ax1.set_title(f'Velocity Profile: v_{velocity_component} vs {direction}')
        ax1.grid(True, alpha=0.3)
        ax1.legend()

        if particle_count is not None:
            ax2 = axes[1]
            ax2.bar(bin_centers, particle_count,
                    width=bin_centers[1] - bin_centers[0], alpha=0.6, color='green')
            ax2.set_xlabel(f'{direction} position')
            ax2.set_ylabel('Particle count in kernel support')
            ax2.set_title('Particles per Bin')
            ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        return fig
    
if __name__ == "__main__":
    # Initialize coarse-grainer
    # Use raw string (r"...") or forward slashes for Windows paths
    vcg = VelocityCoarseGrainer(
        folder_path=r"F:\DEM_DATA\const_V_3D\578_085\pour",
        pattern="dump.stress.*",
        max_frames=10000
    )
    frames = vcg.read_all_frames()
    grid_points, grid_shape = vcg.create_grid(frames[0], n_points=(50,38,10))

### Temporal Coarse Graining

In [ ]:
# Temporal Coarse Graining
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import re
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter  # ADD THIS
import matplotlib

def strain_based_coarse_graining(velocity, strain_rate, dump_freq, time_step,
                                  strain_half_window=0.5, overlap_fraction=0.5):
    """
    Temporally coarse-grain the stress tensor using a Gaussian window centred
    at uniformly spaced strain values.

    The window is specified entirely in STRAIN space so that results are
    independent of dump frequency and timestep once those are fixed.

    Parameters
    ----------
    velocity : ndarray, shape (N_dumps, N_grid, 3, 3)
    strain_rate   : float   [1/s]
    dump_freq     : int     [timesteps per dump]
    time_step     : float   [s]
    strain_half_window : float
        Half-width of the averaging window in strain units.
        Default = 0.5 (average over Δγ = 1.0, centred on each output point).
        The Gaussian sigma is set to strain_half_window / 2, so that weights
        fall to e^{-2} ≈ 0.14 at the window edges.
    overlap_fraction : float in [0, 1)
        Fraction of the window that adjacent output centres share.
        0 = non-overlapping, 0.5 (default) = 50% overlap.
        For ML training, 0 or at most 0.5 is recommended to limit
        autocorrelation between samples.

    Returns
    -------
    cg_stress  : ndarray, shape (N_out, N_grid, 3, 3)
    out_strains: ndarray, shape (N_out,)   strain value at each output centre
    """

    N_dumps = velocity.shape[0]
    dt_dump  = dump_freq * time_step                          # s per dump
    strain_per_dump = dt_dump * strain_rate                   # Δγ per dump

    # Total strain spanned by the simulation
    total_strain = N_dumps * strain_per_dump

    # Gaussian sigma in STRAIN units (not frames)
    #   sigma = half_window / 2  so weights at edges are e^{-2}
    sigma_strain = strain_half_window / 2.0

    # Step between output centres in strain units
    stride_strain = strain_half_window * (1.0 - overlap_fraction)

    print(f"Temporal CG parameters")
    print(f"  dt_dump            = {dt_dump:.4f} s")
    print(f"  strain per dump    = {strain_per_dump:.4f}")
    print(f"  total strain       = {total_strain:.2f}")
    print(f"  window half-width  = {strain_half_window:.3f}  (strain units)")
    print(f"  Gaussian sigma     = {sigma_strain:.3f}  (strain units)")
    print(f"  output stride      = {stride_strain:.3f}  (strain units)")
    print(f"  frames in window   = {strain_half_window / strain_per_dump:.1f}")

    # Output centre strains: from half_window to (total - half_window)
    out_strains = np.arange(strain_half_window,
                            total_strain - strain_half_window + stride_strain,
                            stride_strain)
    print(f"  N output frames    = {len(out_strains)}")

    cg_velocity = np.zeros((len(out_strains),) + velocity.shape[1:],
                         dtype=np.float64)

    # Strain value at each dump
    dump_strains = np.arange(N_dumps) * strain_per_dump   # shape (N_dumps,)

    for k, gamma_centre in enumerate(out_strains):

        #1. Gaussian weights in STRAIN space 
        delta_gamma = dump_strains - gamma_centre           # shape (N_dumps,)
        weights     = np.exp(-0.5 * (delta_gamma / sigma_strain) ** 2)

        # Zero out frames outside 3-sigma support to avoid negligible contributions
        weights[np.abs(delta_gamma) > 3.0 * sigma_strain] = 0.0

        w_sum = weights.sum()
        if w_sum < 1e-12:
            # No frames in window — copy nearest available frame
            nearest = int(np.round(gamma_centre / strain_per_dump))
            nearest = np.clip(nearest, 0, N_dumps - 1)
            cg_velocity[k] = velocity[nearest]
            continue

        weights /= w_sum

        # Weighted sum over all contributing dumps
        # tensordot contracts axis 0 of weights (N_dumps,) with axis 0 of
        # phi (N_dumps, N_grid, 3, 3) -> (N_grid, 3, 3)
        cg_velocity[k] = np.tensordot(weights, velocity, axes=(0, 0))

    print("Temporal CG complete")
    return cg_velocity, out_strains
    
def load_velocity_tensors(folderpath, pattern='velocity_tensor_cg_*.pt', 
                         return_metadata=False):
    """
    Load velocity tensor files and return them sorted by timestep
    
    Parameters:
    -----------
    folderpath : str or Path
        Directory containing tensor files
    pattern : str
        Glob pattern for matching files
    return_metadata : bool
        If True, return timesteps as metadata
        
    Returns:
    --------
    velocity_tensors : torch.Tensor
        Shape (n_timesteps, n_grid_points, 3)
    sorted_timesteps : list
        List of timesteps corresponding to each frame
    """
    folderpath = Path(folderpath)
    
    # Find all matching files
    files = list(folderpath.glob(pattern))
    
    if not files:
        raise FileNotFoundError(f"No files matching '{pattern}' in {folderpath}")
    
    print(f"Found {len(files)} tensor files")
    
    # Extract timesteps and sort files
    file_timestep_pairs = []
    for file in files:
        match = re.search(r'_cg_(\d+)\.pt$', file.name)
        if match:
            timestep = int(match.group(1))
            file_timestep_pairs.append((file, timestep))
        else:
            print(f"Warning: Could not extract timestep from {file.name}, skipping")
    
    if not file_timestep_pairs:
        raise ValueError("No valid timestep information found in filenames")
    
    # Sort by timestep
    file_timestep_pairs.sort(key=lambda x: x[1])
    sorted_files = [f for f, t in file_timestep_pairs]
    sorted_timesteps = [t for f, t in file_timestep_pairs]
    
    print(f"Timestep range: {sorted_timesteps[0]} to {sorted_timesteps[-1]}")
    if len(sorted_timesteps) > 1:
        timestep_interval = sorted_timesteps[1] - sorted_timesteps[0]
        print(f"Timestep interval: {timestep_interval}")
    
    # Get dimensions from first file
    print("\nLoading first file to check dimensions...")
    first_tensor = torch.load(sorted_files[0], weights_only=False)
    
    # Validate tensor structure
    if not isinstance(first_tensor, torch.Tensor):
        raise TypeError(f"Expected torch.Tensor, got {type(first_tensor)}")
    
    velocity_shape = first_tensor.shape # (n_grid_points, 3)
    
    n_grid = velocity_shape[0]
    
    print(f"Grid size: {n_grid} points")
    print(f"Tensor shape per timestep: {velocity_shape}")
    
    # Initialize storage
    num_files = len(sorted_files)
    velocity_tensors = torch.zeros(num_files, n_grid, 3, 
                                   dtype=first_tensor.dtype)
    
    # Store first file data (already loaded)
    velocity_tensors[0] = first_tensor
    
    # Load remaining files
    print("\nLoading remaining files...")
    for i in range(1, num_files):
        # print(f"  Loading {i+1}/{num_files}: timestep {sorted_timesteps[i]}")
        tensor = torch.load(sorted_files[i], weights_only=False)
        # Validate consistency
        if tensor.shape != velocity_shape:
            raise ValueError(f"Shape mismatch at timestep {sorted_timesteps[i]}: "
                           f"expected {velocity_shape}, got {tensor.shape}")
        
        velocity_tensors[i] = tensor
    
    print(f"\n✓ Successfully loaded {num_files} files")
    return velocity_tensors, sorted_timesteps

def plot_velocity_animation_gif(velocity_data, output_filename='velocity_animation.gif',
                                grid_shape=(30, 30, 30), slice_dim=2, slice_index=None,
                                fps=10, dpi=100):
    """
    Create GIF animation of velocity field evolution
    
    Parameters:
    -----------
    velocity_data : torch.Tensor or numpy.ndarray
        Shape: (n_timesteps, n_grid, 3)
        Example: (14, 27000, 3) for velocity components [vx, vy, vz]
    output_filename : str
        Output GIF filename
    grid_shape : tuple
        3D grid dimensions (nx, ny, nz)
        Example: (30, 30, 30) for 27000 points
    slice_dim : int
        Dimension for 2D slice (0=x, 1=y, 2=z)
    slice_index : int or None
        Index for slice (None = middle)
    fps : int
        Frames per second
    dpi : int
        Resolution
    """
    
    # Convert to numpy if needed
    if torch.is_tensor(velocity_data):
        velocity_data = velocity_data.numpy()
    
    n_timesteps = velocity_data.shape[0]
    
    print(f"Creating animation with {n_timesteps} timesteps")
    print(f"Grid shape: {grid_shape}")
    print(f"Velocity data shape: {velocity_data.shape}")
    
    # Validate shape
    expected_grid_size = np.prod(grid_shape)
    if velocity_data.shape[1] != expected_grid_size:
        raise ValueError(f"Grid size mismatch: expected {expected_grid_size}, got {velocity_data.shape[1]}")
    
    # Auto-detect middle slice
    if slice_index is None:
        slice_index = grid_shape[slice_dim] // 2
        print(f"Using middle slice: dim={slice_dim}, index={slice_index}")
    
    # Velocity component names
    components = ['vx', 'vy', 'vz']
    
    # Compute global color scales for consistency
    print("Computing color scales...")
    vmax_list = []
    for comp_idx in range(3):
        all_vals = velocity_data[:, :, comp_idx].flatten()
        non_zero = all_vals[all_vals != 0]
        if len(non_zero) > 0:
            vmax = np.percentile(np.abs(non_zero), 99)
            vmin = np.percentile(non_zero, 1)
        else:
            vmax = np.max(np.abs(all_vals))
            vmin = np.min(all_vals)
        vmax_list.append((vmin, vmax))
        print(f"  {components[comp_idx]}: [{vmin:.6f}, {vmax:.6f}]")
    
    # Compute velocity magnitude for overall visualization
    velocity_mag = np.linalg.norm(velocity_data, axis=2)  # Shape: (n_timesteps, n_grid)
    mag_vmax = np.percentile(velocity_mag[velocity_mag > 0], 99) if np.any(velocity_mag > 0) else 1.0
    print(f"  |v| magnitude: [0, {mag_vmax:.6f}]")
    
    # Determine axis labels based on slice dimension
    if slice_dim == 0:
        xlabel, ylabel = 'y', 'z'
    elif slice_dim == 1:
        xlabel, ylabel = 'x', 'z'
    else:  # slice_dim == 2
        xlabel, ylabel = 'x', 'y'
    
    # Create figure and axes that will be reused
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.ravel()
    
    # Store image objects and colorbars for updating
    images = []
    colorbars = []
    
    # Initialize plots with first frame
    print("Initializing plots...")
    
    # Get first frame data
    velocity_field = velocity_data[0]
    velocity_magnitude = velocity_mag[0]
    
    # Reshape to 3D grid
    velocity_field_3d = velocity_field.reshape(grid_shape + (3,))
    velocity_mag_3d = velocity_magnitude.reshape(grid_shape)
    
    # Extract 2D slice
    if slice_dim == 0:
        velocity_slice = velocity_field_3d[slice_index, :, :, :]
        velocity_mag_slice = velocity_mag_3d[slice_index, :, :]
    elif slice_dim == 1:
        velocity_slice = velocity_field_3d[:, slice_index, :, :]
        velocity_mag_slice = velocity_mag_3d[:, slice_index, :]
    else:  # slice_dim == 2
        velocity_slice = velocity_field_3d[:, :, slice_index, :]
        velocity_mag_slice = velocity_mag_3d[:, :, slice_index]
    
    # Create initial plots for each component
    for ax_idx, comp_name in enumerate(components):
        comp_data = velocity_slice[:, :, ax_idx]
        vmin, vmax = vmax_list[ax_idx]
        
        # Determine colormap and limits
        if vmin < 0 and vmax > 0:
            cmap = 'RdBu_r'
            vmax_sym = max(abs(vmin), abs(vmax))
            vmin_plot, vmax_plot = -vmax_sym, vmax_sym
        else:
            cmap = 'viridis'
            vmin_plot, vmax_plot = vmin, vmax
        
        im = axes[ax_idx].imshow(comp_data.T, 
                                 cmap=cmap, 
                                 origin='lower',
                                 vmin=vmin_plot, 
                                 vmax=vmax_plot,
                                 aspect='auto')
        
        axes[ax_idx].set_title(f'{comp_name} (t=1/{n_timesteps})', fontsize=12)
        axes[ax_idx].set_xlabel(xlabel)
        axes[ax_idx].set_ylabel(ylabel)
        
        cbar = plt.colorbar(im, ax=axes[ax_idx], fraction=0.046, pad=0.04)
        images.append(im)
        colorbars.append(cbar)
    
    # Create velocity magnitude plot
    im = axes[3].imshow(velocity_mag_slice.T, 
                       cmap='plasma', 
                       origin='lower',
                       vmin=0, 
                       vmax=mag_vmax,
                       aspect='auto')
    
    axes[3].set_title(f'|v| Magnitude (t=1/{n_timesteps})', fontsize=12)
    axes[3].set_xlabel(xlabel)
    axes[3].set_ylabel(ylabel)
    
    cbar = plt.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)
    images.append(im)
    colorbars.append(cbar)
    
    fig.suptitle(f'Velocity Field - Timestep 1/{n_timesteps}', fontsize=14, y=0.98)
    plt.tight_layout()
    
    def update_frame(timestep_idx):
        """Update plots for given timestep"""
        
        # Get velocity field for this timestep
        velocity_field = velocity_data[timestep_idx]
        velocity_magnitude = velocity_mag[timestep_idx]
        
        # Reshape to 3D grid
        velocity_field_3d = velocity_field.reshape(grid_shape + (3,))
        velocity_mag_3d = velocity_magnitude.reshape(grid_shape)
        
        # Extract 2D slice
        if slice_dim == 0:
            velocity_slice = velocity_field_3d[slice_index, :, :, :]
            velocity_mag_slice = velocity_mag_3d[slice_index, :, :]
        elif slice_dim == 1:
            velocity_slice = velocity_field_3d[:, slice_index, :, :]
            velocity_mag_slice = velocity_mag_3d[:, slice_index, :]
        else:  # slice_dim == 2
            velocity_slice = velocity_field_3d[:, :, slice_index, :]
            velocity_mag_slice = velocity_mag_3d[:, :, slice_index]
        
        # Update each velocity component
        for ax_idx, comp_name in enumerate(components):
            comp_data = velocity_slice[:, :, ax_idx]
            images[ax_idx].set_data(comp_data.T)
            axes[ax_idx].set_title(f'{comp_name} (t={timestep_idx+1}/{n_timesteps})', fontsize=12)
        
        # Update velocity magnitude
        images[3].set_data(velocity_mag_slice.T)
        axes[3].set_title(f'|v| Magnitude (t={timestep_idx+1}/{n_timesteps})', fontsize=12)
        
        # Update main title
        fig.suptitle(f'Velocity Field - Timestep {timestep_idx+1}/{n_timesteps}', 
                     fontsize=14, y=0.98)
        
        return images
    
    # Create animation using FuncAnimation
    print(f"\nGenerating animation with {n_timesteps} frames...")
    anim = FuncAnimation(fig, update_frame, frames=n_timesteps, 
                        interval=1000/fps, blit=False, repeat=True)
    
    # Save as GIF
    writer = PillowWriter(fps=fps)
    anim.save(output_filename, writer=writer, dpi=dpi)
    
    plt.close(fig)
    
    print(f"\n✓ Animation saved: {output_filename}")
    print(f"  File size: {Path(output_filename).stat().st_size / (1024**2):.2f} MB")
    print(f"  Duration: {n_timesteps/fps:.1f} seconds")
    
    return output_filename


if __name__ == "__main__":

    folder = r"F:\DEM_DATA\const_V_3D\vf578vt085\pour"
    vt_match = re.search(r'vt(\d+)', folder)
    if vt_match:
        vt_value = vt_match.group(1) 
    else:
        print(r"No top wall velocity found")
        vt_value = '0'
    vt_to_strain_rate = {'014': 0.035, '003': 0.00775, '03': 0.0775, '085': 0.2125, '14': 0.35, '0': 0.0}
    strain_rate = vt_to_strain_rate[vt_value]
    
    velocity_tensor, time_steps = load_velocity_tensors(folderpath=r"D:\ml_granular\Sparse Modeling\velocity_cgs")
    time_step = 5.18e-6
    dump_freq = 1e5
    cg_velocities,_ = strain_based_coarse_graining(velocity_tensor, strain_rate, dump_freq, time_step)